In [7]:
import pandas as pd
df = pd.read_csv(r"D:\Phishing Scam Detection\data\SMSSpamCollection", sep='\t', header=None, names=['label', 'message'])

print(df.head())

print("\nShape:", df.shape)
print("\nColumns:", df.columns)

print("\nClass distribution:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicates:")
print(df.duplicated().sum())

df = df.drop_duplicates()

print("Shape after removing duplicates:", df.shape)
print("Duplicates remaining:", df.duplicated().sum())

df["label"] = df["label"].map({"ham": 0, "spam": 1})

print(df.head())
print("\nLabel distribution:")
print(df["label"].value_counts())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...

Shape: (5572, 2)

Columns: Index(['label', 'message'], dtype='str')

Class distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

Missing values:
label      0
message    0
dtype: int64

Duplicates:
403
Shape after removing duplicates: (5169, 2)
Duplicates remaining: 0
   label                                            message
0      0  Go until jurong point, crazy.. Available only ...
1      0                      Ok lar... Joking wif u oni...
2      1  Free entry in 2 a wkly comp to win FA Cup fina...
3      0  U dun say so early hor... U c already then say...
4      0  Nah I don't think he goes to usf, he lives aro..

In [8]:
from sklearn.model_selection import train_test_split

X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))
print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())


X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training messages:", len(X_train))
print("Testing messages:", len(X_test))
print("\nTraining label distribution:")
print(y_train.value_counts())

print("\nTesting label distribution:")
print(y_test.value_counts())

Training messages: 4135
Testing messages: 1034

Training label distribution:
label
0    3613
1     522
Name: count, dtype: int64

Testing label distribution:
label
0    903
1    131
Name: count, dtype: int64
Training messages: 4135
Testing messages: 1034

Training label distribution:
label
0    3613
1     522
Name: count, dtype: int64

Testing label distribution:
label
0    903
1    131
Name: count, dtype: int64


In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("Training matrix shape:", X_train_tfidf.shape)
print("Testing matrix shape:", X_test_tfidf.shape)

Training matrix shape: (4135, 7414)
Testing matrix shape: (1034, 7414)


In [10]:
from sklearn.linear_model import LogisticRegression

model_balanced = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

model_balanced.fit(X_train_tfidf, y_train)

y_pred_balanced = model_balanced.predict(X_test_tfidf)

print("Balanced model trained!")

Balanced model trained!


In [11]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_balanced))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_balanced,
    target_names=["Ham", "Spam"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))

Accuracy: 0.9738878143133463

Classification Report:
              precision    recall  f1-score   support

         Ham       0.98      0.99      0.99       903
        Spam       0.91      0.89      0.90       131

    accuracy                           0.97      1034
   macro avg       0.94      0.94      0.94      1034
weighted avg       0.97      0.97      0.97      1034


Confusion Matrix:
[[891  12]
 [ 15 116]]


In [12]:
from sklearn.naive_bayes import MultinomialNB

nb_model = MultinomialNB()

nb_model.fit(X_train_tfidf, y_train)

y_pred_nb = nb_model.predict(X_test_tfidf)

print("Naive Bayes model trained!")

Naive Bayes model trained!


In [13]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_nb,
    target_names=["Ham", "Spam"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_nb))

Accuracy: 0.9671179883945842

Classification Report:
              precision    recall  f1-score   support

         Ham       0.96      1.00      0.98       903
        Spam       1.00      0.74      0.85       131

    accuracy                           0.97      1034
   macro avg       0.98      0.87      0.92      1034
weighted avg       0.97      0.97      0.96      1034


Confusion Matrix:
[[903   0]
 [ 34  97]]


In [14]:
import re
import pandas as pd

def extract_features(messages):
    features = pd.DataFrame(index=messages.index)

    features["message_length"] = messages.str.len()
    features["word_count"] = messages.str.split().str.len()
    features["digit_count"] = messages.str.count(r"\d")
    features["exclamation_count"] = messages.str.count("!")
    features["question_count"] = messages.str.count(r"\?")
    features["currency_count"] = messages.str.count(r"[£$€₹]")
    features["url_count"] = messages.str.count(
        r"https?://|www\.|bit\.ly|tinyurl"
    )
    features["phone_number_count"] = messages.str.count(
        r"\b\d{7,}\b"
    )

    return features

train_features = extract_features(X_train)
test_features = extract_features(X_test)

print(train_features.head())

      message_length  word_count  digit_count  exclamation_count  \
336               45          10            0                  1   
390               47           9            0                  0   
582               46          10            0                  0   
1387              27           7            1                  0   
1095              71          17            0                  0   

      question_count  currency_count  url_count  phone_number_count  
336                1               0          0                   0  
390                1               0          0                   0  
582                0               0          0                   0  
1387               0               0          0                   0  
1095               0               0          0                   0  


In [15]:
from scipy.sparse import hstack

X_train_combined = hstack([
    X_train_tfidf,
    train_features
])

X_test_combined = hstack([
    X_test_tfidf,
    test_features
])

print("Combined training shape:", X_train_combined.shape)
print("Combined testing shape:", X_test_combined.shape)

Combined training shape: (4135, 7422)
Combined testing shape: (1034, 7422)


In [16]:
from sklearn.linear_model import LogisticRegression

combined_model = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

combined_model.fit(X_train_combined, y_train)

y_pred_combined = combined_model.predict(X_test_combined)

print("Combined model trained!")

Combined model trained!


d:\Phishing Scam Detection\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [17]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Accuracy:", accuracy_score(y_test, y_pred_combined))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_combined,
    target_names=["Ham", "Spam"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_combined))

Accuracy: 0.9738878143133463

Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99       903
        Spam       0.89      0.91      0.90       131

    accuracy                           0.97      1034
   macro avg       0.94      0.95      0.94      1034
weighted avg       0.97      0.97      0.97      1034


Confusion Matrix:
[[888  15]
 [ 12 119]]


In [18]:
y_prob = combined_model.predict_proba(X_test_combined)[:, 1]

print("First 10 spam probabilities:")
print(y_prob[:10])

First 10 spam probabilities:
[0.02453626 0.0476696  0.05960932 0.04421687 0.06123576 0.13372576
 0.09041085 0.01446471 0.02011126 0.03279485]


In [19]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

for threshold in thresholds:
    y_pred_threshold = (y_prob >= threshold).astype(int)

    precision = precision_score(y_test, y_pred_threshold)
    recall = recall_score(y_test, y_pred_threshold)
    f1 = f1_score(y_test, y_pred_threshold)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.2f} | "
        f"Recall: {recall:.2f} | "
        f"F1: {f1:.2f}"
    )

Threshold: 0.30 | Precision: 0.84 | Recall: 0.94 | F1: 0.88
Threshold: 0.35 | Precision: 0.86 | Recall: 0.94 | F1: 0.90
Threshold: 0.40 | Precision: 0.86 | Recall: 0.92 | F1: 0.89
Threshold: 0.45 | Precision: 0.86 | Recall: 0.92 | F1: 0.89
Threshold: 0.50 | Precision: 0.89 | Recall: 0.91 | F1: 0.90
Threshold: 0.55 | Precision: 0.92 | Recall: 0.90 | F1: 0.91
Threshold: 0.60 | Precision: 0.92 | Recall: 0.90 | F1: 0.91
Threshold: 0.65 | Precision: 0.93 | Recall: 0.90 | F1: 0.91
Threshold: 0.70 | Precision: 0.93 | Recall: 0.89 | F1: 0.91


In [20]:
from sklearn.model_selection import train_test_split

X_train_final, X_val, y_train_final, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=42,
    stratify=y_train
)

print("Training:", len(X_train_final))
print("Validation:", len(X_val))
print("Test:", len(X_test))

Training: 3308
Validation: 827
Test: 1034


In [21]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer_final = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_train_final_tfidf = vectorizer_final.fit_transform(X_train_final)

X_val_tfidf = vectorizer_final.transform(X_val)

X_test_final_tfidf = vectorizer_final.transform(X_test)

print("Training:", X_train_final_tfidf.shape)
print("Validation:", X_val_tfidf.shape)
print("Test:", X_test_final_tfidf.shape)

Training: (3308, 6450)
Validation: (827, 6450)
Test: (1034, 6450)


In [22]:
train_features_final = extract_features(X_train_final)
val_features_final = extract_features(X_val)
test_features_final = extract_features(X_test)

print("Training engineered features:", train_features_final.shape)
print("Validation engineered features:", val_features_final.shape)
print("Test engineered features:", test_features_final.shape)

Training engineered features: (3308, 8)
Validation engineered features: (827, 8)
Test engineered features: (1034, 8)


In [23]:
from scipy.sparse import hstack

X_train_final_combined = hstack([
    X_train_final_tfidf,
    train_features_final
])

X_val_combined = hstack([
    X_val_tfidf,
    val_features_final
])

X_test_final_combined = hstack([
    X_test_final_tfidf,
    test_features_final
])

print("Training:", X_train_final_combined.shape)
print("Validation:", X_val_combined.shape)
print("Test:", X_test_final_combined.shape)

Training: (3308, 6458)
Validation: (827, 6458)
Test: (1034, 6458)


In [24]:
from sklearn.linear_model import LogisticRegression

final_model = LogisticRegression(
    class_weight="balanced",
    random_state=42
)

final_model.fit(X_train_final_combined, y_train_final)

print("Final development model trained!")

Final development model trained!


d:\Phishing Scam Detection\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [25]:
y_val_prob = final_model.predict_proba(X_val_combined)[:, 1]

print("First 10 validation probabilities:")
print(y_val_prob[:10])

First 10 validation probabilities:
[0.01628382 0.99999097 0.03545298 0.01511815 0.87132578 0.02330293
 0.08205125 0.0251756  0.03963265 0.03951427]


In [26]:
from sklearn.metrics import precision_score, recall_score, f1_score

thresholds = [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]

for threshold in thresholds:
    y_val_pred = (y_val_prob >= threshold).astype(int)

    precision = precision_score(y_val, y_val_pred)
    recall = recall_score(y_val, y_val_pred)
    f1 = f1_score(y_val, y_val_pred)

    print(
        f"Threshold: {threshold:.2f} | "
        f"Precision: {precision:.2f} | "
        f"Recall: {recall:.2f} | "
        f"F1: {f1:.2f}"
    )

Threshold: 0.30 | Precision: 0.87 | Recall: 0.96 | F1: 0.91
Threshold: 0.35 | Precision: 0.89 | Recall: 0.96 | F1: 0.93
Threshold: 0.40 | Precision: 0.91 | Recall: 0.95 | F1: 0.93
Threshold: 0.45 | Precision: 0.93 | Recall: 0.95 | F1: 0.94
Threshold: 0.50 | Precision: 0.93 | Recall: 0.95 | F1: 0.94
Threshold: 0.55 | Precision: 0.94 | Recall: 0.95 | F1: 0.95
Threshold: 0.60 | Precision: 0.95 | Recall: 0.93 | F1: 0.94
Threshold: 0.65 | Precision: 0.96 | Recall: 0.90 | F1: 0.93
Threshold: 0.70 | Precision: 0.96 | Recall: 0.90 | F1: 0.93


In [27]:
y_test_prob = final_model.predict_proba(X_test_final_combined)[:, 1]

y_test_final = (y_test_prob >= 0.55).astype(int)

print("Final test predictions generated!")

Final test predictions generated!


In [28]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

print("Final Test Accuracy:", accuracy_score(y_test, y_test_final))

print("\nFinal Test Classification Report:")
print(classification_report(
    y_test,
    y_test_final,
    target_names=["Ham", "Spam"]
))

print("\nFinal Test Confusion Matrix:")
print(confusion_matrix(y_test, y_test_final))

Final Test Accuracy: 0.9787234042553191

Final Test Classification Report:
              precision    recall  f1-score   support

         Ham       0.99      0.99      0.99       903
        Spam       0.92      0.91      0.92       131

    accuracy                           0.98      1034
   macro avg       0.95      0.95      0.95      1034
weighted avg       0.98      0.98      0.98      1034


Final Test Confusion Matrix:
[[893  10]
 [ 12 119]]


In [ ]:
# Check the important ML objects currently available

objects_to_check = [
    "final_model",
    "tfidf_vectorizer",
    "tfidf",
    "vectorizer",
    "threshold"
]

for obj in objects_to_check:
    if obj in globals():
        print(f"✓ {obj} is available")
    else:
        print(f"✗ {obj} is not available")


✓ final_model is available
✗ tfidf_vectorizer is not available
✗ tfidf is not available
✓ vectorizer is available
✓ threshold is available


In [33]:
# Set the final threshold selected during validation

threshold = 0.55

print("Final threshold:", threshold)

Final threshold: 0.55


In [34]:
print("Model:", type(final_model))
print("Vectorizer:", type(vectorizer))
print("Threshold:", threshold)

Model: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Vectorizer: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
Threshold: 0.55


In [35]:
import joblib
from pathlib import Path

# Create the models folder if it doesn't already exist
models_dir = Path("../models")
models_dir.mkdir(exist_ok=True)

# Save the trained model
joblib.dump(final_model, models_dir / "phishing_scam_model.pkl")

# Save the TF-IDF vectorizer
joblib.dump(vectorizer, models_dir / "tfidf_vectorizer.pkl")

# Save the decision threshold
joblib.dump(threshold, models_dir / "threshold.pkl")

print("✓ Model saved")
print("✓ Vectorizer saved")
print("✓ Threshold saved")

✓ Model saved
✓ Vectorizer saved
✓ Threshold saved


In [36]:
import re
import numpy as np
import pandas as pd


def extract_features(messages):
    features = pd.DataFrame()

    features["message_length"] = messages.apply(len)

    features["word_count"] = messages.apply(
        lambda x: len(x.split())
    )

    features["digit_count"] = messages.apply(
        lambda x: sum(c.isdigit() for c in x)
    )

    features["exclamation_count"] = messages.apply(
        lambda x: x.count("!")
    )

    features["question_count"] = messages.apply(
        lambda x: x.count("?")
    )

    features["currency_count"] = messages.apply(
        lambda x: len(re.findall(r"[₹$€£]", x))
    )

    features["url_count"] = messages.apply(
        lambda x: len(
            re.findall(
                r"(https?://\S+|www\.\S+)",
                x,
                re.IGNORECASE
            )
        )
    )

    features["phone_number_count"] = messages.apply(
        lambda x: len(
            re.findall(
                r"\b\d{7,15}\b",
                x
            )
        )
    )

    return features

In [42]:
from scipy.sparse import hstack


def predict_message(message):
    # Convert the message into a pandas Series
    message_series = pd.Series([message])

    # Use the FINAL TF-IDF vectorizer
    tfidf_features = vectorizer_final.transform(message_series)

    # Create the 8 engineered features
    engineered_features = extract_features(message_series)

    # Convert engineered features to a NumPy array
    engineered_features = engineered_features.values

    # Combine TF-IDF + engineered features
    combined_features = hstack([
        tfidf_features,
        engineered_features
    ])

    # Get probability that the message is spam/scam
    probability = final_model.predict_proba(
        combined_features
    )[0][1]

    # Apply the final threshold
    prediction = 1 if probability >= threshold else 0

    if prediction == 1:
        result = "SCAM / SPAM"
    else:
        result = "SAFE"

    return result, probability

In [43]:
message = "Congratulations! You have won ₹50,000. Click https://example.com to claim your prize."

result, probability = predict_message(message)

print("Message:", message)
print("Prediction:", result)
print("Probability:", round(probability * 100, 2), "%")

Message: Congratulations! You have won ₹50,000. Click https://example.com to claim your prize.
Prediction: SCAM / SPAM
Probability: 99.96 %


In [49]:
def detect_warning_signs(message):
    warnings = []

    # Convert message to lowercase for easier checking
    text = message.lower()

    # Check for URLs
    if re.search(r"(https?://\S+|www\.\S+)", message, re.IGNORECASE):
        warnings.append("Contains a URL")

    # Check for currency symbols
    if re.search(r"[₹$€£]", message):
        warnings.append("Contains financial/currency information")

    # Check for phone numbers
    if re.search(r"\b\d{7,15}\b", message):
        warnings.append("Contains a phone number")

    # Check for urgency-related words
    urgency_words = [
        "urgent",
        "immediately",
        "act now",
        "hurry",
        "expires",
        "limited time",
        "asap"
    ]

    if any(word in text for word in urgency_words):
        warnings.append("Uses urgency-related language")

    # Check for prize/reward language
    reward_words = [
        "won",
        "winner",
        "prize",
        "reward",
        "lottery",
        "congratulations",
        "cash prize",
        "free"
    ]

    if any(word in text for word in reward_words):
        warnings.append("Contains prize/reward language")

    # Check for account/security language
    security_words = [
        "verify your account",
        "verify account",
        "password",
        "otp",
        "bank account",
        "credit card",
        "debit card",
        "login"
    ]

    if any(word in text for word in security_words):
        warnings.append("Requests or mentions sensitive account information")

    return warnings

In [50]:
message = "Hey, are we still meeting at the college tomorrow? Let me know."

warnings = detect_warning_signs(message)

print("Warning signs detected:")

if warnings:
    for warning in warnings:
        print("•", warning)
else:
    print("• No obvious warning signs detected")

Warning signs detected:
• No obvious warning signs detected


In [53]:
def analyze_message(message):
    # Get ML prediction and probability
    result, probability = predict_message(message)

    # Detect human-readable warning signs
    warnings = detect_warning_signs(message)

    # Convert probability to percentage
    risk_percentage = round(probability * 100, 2)

    return {
        "prediction": result,
        "risk_percentage": risk_percentage,
        "warning_signs": warnings
    }

In [54]:
message = "Congratulations! You have won ₹50,000. Click https://example.com to claim your prize."

analysis = analyze_message(message)

print("Prediction:", analysis["prediction"])
print("Risk:", analysis["risk_percentage"], "%")

print("\nWarning signs:")

if analysis["warning_signs"]:
    for warning in analysis["warning_signs"]:
        print("•", warning)
else:
    print("• No obvious warning signs detected")

Prediction: SCAM / SPAM
Risk: 99.96 %

Warning signs:
• Contains a URL
• Contains financial/currency information
• Contains prize/reward language


In [55]:
import joblib
from pathlib import Path

models_dir = Path("../models")

loaded_model = joblib.load(models_dir / "phishing_scam_model.pkl")
loaded_vectorizer = joblib.load(models_dir / "tfidf_vectorizer.pkl")
loaded_threshold = joblib.load(models_dir / "threshold.pkl")

print("✓ Model loaded")
print("✓ Vectorizer loaded")
print("✓ Threshold loaded")

print("\nModel:", type(loaded_model))
print("Vectorizer:", type(loaded_vectorizer))
print("Threshold:", loaded_threshold)

✓ Model loaded
✓ Vectorizer loaded
✓ Threshold loaded

Model: <class 'sklearn.linear_model._logistic.LogisticRegression'>
Vectorizer: <class 'sklearn.feature_extraction.text.TfidfVectorizer'>
Threshold: 0.55


In [56]:
import joblib
from pathlib import Path

models_dir = Path("../models")

# Save the CORRECT final vectorizer
joblib.dump(
    vectorizer_final,
    models_dir / "tfidf_vectorizer.pkl"
)

print("✓ Correct final vectorizer saved")
print(
    "Number of TF-IDF features:",
    len(vectorizer_final.get_feature_names_out())
)

✓ Correct final vectorizer saved
Number of TF-IDF features: 6450


In [57]:
loaded_vectorizer = joblib.load(
    models_dir / "tfidf_vectorizer.pkl"
)

print(
    "Saved vectorizer features:",
    len(loaded_vectorizer.get_feature_names_out())
)

Saved vectorizer features: 6450
